# Measuring Internet Access

In [1]:
import os
import chardet
import pandas as pd
import math

from my_functions import pad_column_values

## Setup and Import FCC Data

In [2]:
current_directory = os.getcwd()

# Define the file path and chunk size
file_path = "data_inputs/fbd_us_without_satellite_dec2021_v1.csv"
chunk_size = 100000  # Adjust based on your needs

# Attempt to read the CSV in chunks with ISO-8859-1 encoding, skipping bad lines
chunks = []
try:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size, encoding='ISO-8859-1', on_bad_lines='skip'):
        chunks.append(chunk)

    # Optionally, concatenate chunks into a single DataFrame
    data = pd.concat(chunks, ignore_index=True)
    print("File loaded successfully.")
except UnicodeDecodeError as e:
    print(f"UnicodeDecodeError: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

File loaded successfully.


### Pad `BlockCode` Column

In [3]:
data = pad_column_values(data, "BlockCode", target_length = 15)


Ensuring 'BlockCode' values are treated as strings.
Character counts for 'BlockCode' before padding:
BlockCode
14     4412858
15    24785479
Name: count, dtype: int64
Applied padding to 'BlockCode' using '0' to reach 15 characters.
All 'BlockCode' values are now exactly 15 characters long.



### Drop Unnecessary Columns

In [4]:
# Add "BlockCode" variable — used because of other code that was already written

# List of columns you want to drop
columns_to_drop = ["Provider_Id", "FRN", "ProviderName", "DBAName", "HoldingCompanyName", "HocoFinal", "StateAbbr"] # Notice "HocoNum" not included to allow for identifier

# Drop the selected columns
data = data.drop(columns=columns_to_drop)

# Sort the data by 'BlockCode'
data = data.sort_values(by=["BlockCode"])

data

,LogRecNo,HocoNum,BlockCode,TechCode,Consumer,MaxAdDown,MaxAdUp,Business
8574342,8574343,131480,010010201001000,42,1,1000.0,50.0,0
21373483,29444380,130403,010010201001000,70,1,25.0,3.0,1
17098575,25169472,130235,010010201001000,43,1,1000.0,35.0,0
4465063,4465064,130077,010010201001000,11,1,18.0,1.5,0
21373478,29444375,130403,010010201001001,70,1,25.0,3.0,1
...,...,...,...,...,...,...,...,...
13116677,21187574,130741,780309900000004,70,1,25.0,5.0,1
13116676,21187573,130741,780309900000005,70,1,25.0,5.0,1
13116675,21187572,130741,780309900000006,70,1,25.0,5.0,1
13116674,21187571,130741,780309900000007,70,1,25.0,5.0,1


### Add variables that describe different levels of internet speed


- `Zoom_Min_5_5_MBPS`: Describes whether the block contains `5` mbps download and `5` mbps upload speed. Both are commonly used metrics of minimum speeds required for a Zoom call. <br><br>
    > `1` if `MaxAdDown` equal or above `5` and `MaxAdDown` equal or above `5`. `0` otherwise  <br>
- `FCC_Def_High_Speed_25_3_MBPS`: Describes whether the block contains `25` mbps download and `3` mbps upload speed. These metrics are used by the FCC to define high-speed internet access. Note, the FCC has received criticism by multiple organiztions and experts regarding these metrics. <br><br>
    > `1` if `MaxAdDown`equal or above `25` and `MaxAdDown` equal or above `3`. `0` otherwise  <br>
- `Other_Def_High_Speed_100_20_MBPS`: Describes whether the block contains `100` mbps download and `20` mbps upload speed. These metrics are commonly used organizations and experts to describe high-speed internet access  <br><br>
    > `1` if `MaxAdDown`equal or above `100` and `MaxAdDown` equal or above `20`. `0` otherwise  <br>

In [5]:
# Creating the new variables based on the conditions provided
data['Zoom_Min_5_5_MBPS'] = ((data['MaxAdDown'] >= 5) & (data['MaxAdDown'] >= 5)).astype(int)
data['FCC_Def_High_Speed_25_3_MBPS'] = ((data['MaxAdDown'] >= 25) & (data['MaxAdDown'] >= 3)).astype(int)
data['Other_Def_High_Speed_100_20_MBPS'] = ((data['MaxAdDown'] >= 100) & (data['MaxAdDown'] >= 20)).astype(int)

# Split the data into components
data['StateFIPS'] = data['BlockCode'].str[:2]  # First 2 characters for State level analyses
data['County'] = data['BlockCode'].str[:5]  # First 5 characters for County level analyses
data['CensusTract'] = data['BlockCode'].str[:11]  # First 11 characters for Census Tract level Analyses
data['CensusBlock'] = data['BlockCode'].str[11:]  # Last 4 characters for Census Block

data

,LogRecNo,HocoNum,BlockCode,TechCode,Consumer,MaxAdDown,MaxAdUp,Business,Zoom_Min_5_5_MBPS,FCC_Def_High_Speed_25_3_MBPS,Other_Def_High_Speed_100_20_MBPS,StateFIPS,County,CensusTract,CensusBlock
8574342,8574343,131480,010010201001000,42,1,1000.0,50.0,0,1,1,1,01,01001,01001020100,1000
21373483,29444380,130403,010010201001000,70,1,25.0,3.0,1,1,1,0,01,01001,01001020100,1000
17098575,25169472,130235,010010201001000,43,1,1000.0,35.0,0,1,1,1,01,01001,01001020100,1000
4465063,4465064,130077,010010201001000,11,1,18.0,1.5,0,1,0,0,01,01001,01001020100,1000
21373478,29444375,130403,010010201001001,70,1,25.0,3.0,1,1,1,0,01,01001,01001020100,1001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13116677,21187574,130741,780309900000004,70,1,25.0,5.0,1,1,1,0,78,78030,78030990000,0004
13116676,21187573,130741,780309900000005,70,1,25.0,5.0,1,1,1,0,78,78030,78030990000,0005
13116675,21187572,130741,780309900000006,70,1,25.0,5.0,1,1,1,0,78,78030,78030990000,0006
13116674,21187571,130741,780309900000007,70,1,25.0,5.0,1,1,1,0,78,78030,78030990000,0007


## Import the `School_District_to_Census_Block` File

In [6]:
# Define the file path
file_path = "data_inputs/LEAIDtoCensusTract_XWalk.xlsx"

# Read the Excel file
crosswalk_data = pd.read_excel(file_path)

crosswalk_data

,LEAID,NAME_LEA21,TRACT,COUNT,LANDAREA,WATERAREA
0,100001,Fort Rucker School District,1031010300,2,23.428369,0.000000
1,100001,Fort Rucker School District,1045020000,2,66.513404,1.081745
2,100003,Maxwell AFB School District,1101000900,2,3.356590,0.143795
3,100003,Maxwell AFB School District,1101001000,2,0.001526,0.000000
4,100005,Albertville City School District,1095030701,9,2.125782,0.000000
...,...,...,...,...,...,...
126923,7800030,Virgin Islands Department of Education,78030960900,32,3.127506,1.173777
126924,7800030,Virgin Islands Department of Education,78030961000,32,0.772173,0.910606
126925,7800030,Virgin Islands Department of Education,78030961100,32,1.343339,0.000000
126926,7800030,Virgin Islands Department of Education,78030961200,32,0.425224,0.309706


### Pad `BlockCode` and `LEADID` variables

In [7]:
# Copy and pad variables while changing name
crosswalk_data["LEADID"] = crosswalk_data["LEAID"]
crosswalk_data = pad_column_values(crosswalk_data, "LEADID", target_length = 7)

crosswalk_data["CensusTract"] = crosswalk_data["TRACT"]
crosswalk_data = pad_column_values(crosswalk_data, "CensusTract", target_length = 11)

# Drop the selected columns
columns_to_drop = ["LEAID", "TRACT"]
crosswalk_data = crosswalk_data.drop(columns = columns_to_drop)

# Display the data
crosswalk_data

Ensuring 'LEADID' values are treated as strings.
Character counts for 'LEADID' before padding:
LEADID
6     26018
7    100910
Name: count, dtype: int64
Applied padding to 'LEADID' using '0' to reach 7 characters.
All 'LEADID' values are now exactly 7 characters long.

Ensuring 'CensusTract' values are treated as strings.
Character counts for 'CensusTract' before padding:
CensusTract
10     26018
11    100910
Name: count, dtype: int64
Applied padding to 'CensusTract' using '0' to reach 11 characters.
All 'CensusTract' values are now exactly 11 characters long.



,NAME_LEA21,COUNT,LANDAREA,WATERAREA,LEADID,CensusTract
0,Fort Rucker School District,2,23.428369,0.000000,0100001,01031010300
1,Fort Rucker School District,2,66.513404,1.081745,0100001,01045020000
2,Maxwell AFB School District,2,3.356590,0.143795,0100003,01101000900
3,Maxwell AFB School District,2,0.001526,0.000000,0100003,01101001000
4,Albertville City School District,9,2.125782,0.000000,0100005,01095030701
...,...,...,...,...,...,...
126923,Virgin Islands Department of Education,32,3.127506,1.173777,7800030,78030960900
126924,Virgin Islands Department of Education,32,0.772173,0.910606,7800030,78030961000
126925,Virgin Islands Department of Education,32,1.343339,0.000000,7800030,78030961100
126926,Virgin Islands Department of Education,32,0.425224,0.309706,7800030,78030961200


### Merge Data to get Internet Speeds for Each School District at Census Block Level

In [8]:
# Merge data
df = pd.merge(crosswalk_data, data, on="CensusTract", how='inner').drop_duplicates()

df = df.sort_values("CensusTract")

df

,NAME_LEA21,COUNT,LANDAREA,WATERAREA,LEADID,CensusTract,LogRecNo,HocoNum,BlockCode,TechCode,Consumer,MaxAdDown,MaxAdUp,Business,Zoom_Min_5_5_MBPS,FCC_Def_High_Speed_25_3_MBPS,Other_Def_High_Speed_100_20_MBPS,StateFIPS,County,CensusBlock
83032,Autauga County School District,17,3.793571,0.010979,0100240,01001020100,25169438,130235,010010201002011,43,1,1000.0,35.0,0,1,1,1,01,01001,2011
82986,Autauga County School District,17,3.793571,0.010979,0100240,01001020100,4465048,130077,010010201001022,11,1,12.0,1.5,0,1,0,0,01,01001,1022
82987,Autauga County School District,17,3.793571,0.010979,0100240,01001020100,7516096,420017,010010201001022,50,1,1000.0,1000.0,1,1,1,1,01,01001,1022
82988,Autauga County School District,17,3.793571,0.010979,0100240,01001020100,29444936,130403,010010201001022,70,1,25.0,3.0,1,1,1,0,01,01001,1022
82989,Autauga County School District,17,3.793571,0.010979,0100240,01001020100,8574333,131480,010010201001022,42,1,1000.0,50.0,0,1,1,1,01,01001,1022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56009537,Virgin Islands Department of Education,32,0.000000,265.382518,7800030,78030990000,21187575,130741,780309900000003,70,1,25.0,5.0,1,1,1,0,78,78030,0003
56009538,Virgin Islands Department of Education,32,0.000000,265.382518,7800030,78030990000,21187574,130741,780309900000004,70,1,25.0,5.0,1,1,1,0,78,78030,0004
56009539,Virgin Islands Department of Education,32,0.000000,265.382518,7800030,78030990000,21187573,130741,780309900000005,70,1,25.0,5.0,1,1,1,0,78,78030,0005
56009540,Virgin Islands Department of Education,32,0.000000,265.382518,7800030,78030990000,21187572,130741,780309900000006,70,1,25.0,5.0,1,1,1,0,78,78030,0006


### Split `BlockCode` into Multiple Variables

In [9]:
print("Number of Unique LEADID values:", df["LEADID"].nunique())
print()
print("Number of Unique School Disrict Names:", df["NAME_LEA21"].nunique())

Number of Unique LEADID values: 13260

Number of Unique School Disrict Names: 12679


### Save Data before Aggregating Up

In [10]:
# Define the number of files to split into
num_files = 2 # Change this to 2, 3, 4, 5, or any other number

# Directory to save the output files
output_dir = 'data_outputs/internet_access_nonaggregated'
os.makedirs(output_dir, exist_ok=True)

# Calculate the size of each split
chunk_size = math.ceil(len(df) / num_files)

# Split and save the data
for i in range(num_files):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(df))  # Ensure we don't exceed the DataFrame size
    output_file = f'{output_dir}/part{i + 1}.csv'
    df.iloc[start_idx:end_idx].to_csv(output_file, index=False)
    print(f"Saved: {output_file}")

print("Data has been split and saved successfully.")

Saved: data_outputs/internet_access_nonaggregated/part1.csv
Saved: data_outputs/internet_access_nonaggregated/part2.csv
Data has been split and saved successfully.


## Aggregate Data Up by Different Levels
1. School District
2. Census Tract
3. County
4. State

In [11]:

def aggregate_data(df, group_by_columns, output_file):
    """
    Aggregates the internet access data at specified group levels and saves the output to a CSV file.
    
    Args:
        df (pd.DataFrame): The input data frame.
        group_by_columns (list): List of column names to group by.
        output_file (str): The file path to save the aggregated data.
    """
    aggregated_data = df.groupby(group_by_columns).agg(
        n_school_districts=('NAME_LEA21', 'nunique'),
        n_counties=('County', 'nunique'),
        n_tracts=('CensusTract', 'nunique'),
        n_blocks=('CensusBlock', 'nunique'),
        n_ISPs=('HocoNum', 'nunique'),
        #
        zoom_5_5_coverage_pct=("Zoom_Min_5_5_MBPS", "mean"),
        fcc_25_3_coverage_pct=("FCC_Def_High_Speed_25_3_MBPS", "mean"),
        other_100_20_coverage_pct=('Other_Def_High_Speed_100_20_MBPS', "mean"),
        #
        max_MaxAdDown=('MaxAdDown', 'max'),
        median_MaxAdDown=('MaxAdDown', 'median'),
        mean_MaxAdDown=('MaxAdDown', 'mean'),
        min_MaxAdDown=('MaxAdDown', 'min'),
        std_MaxAdDown=('MaxAdDown', 'std'),
        #
        max_MaxAdUp=('MaxAdUp', 'max'),
        median_MaxAdUp=('MaxAdUp', 'median'),
        mean_MaxAdUp=('MaxAdUp', 'mean'),
        min_MaxAdUp=('MaxAdUp', 'min'),
        std_MaxAdUp=('MaxAdUp', 'std'),
        #
        n_LEADIDs=('LEADID', 'nunique'),
    ).reset_index()

    # Save to CSV
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    aggregated_data.to_csv(output_file, index=False)

In [12]:
# List of grouping levels and their corresponding output files
grouping_levels = {
    "school_district_name": ["NAME_LEA21", "StateFIPS"],
    "school_district_LEADID": ["LEADID", "StateFIPS"],
    "census_tract_level": ["CensusTract", "StateFIPS"],
    "county_level": ["County", "StateFIPS"],
    "state_level": ["StateFIPS"],
}

# Loop through each grouping level and generate aggregated data
for level_name, group_by_cols in grouping_levels.items():
    output_file = f"data_outputs/internet_access_aggregated/{level_name}.csv"
    aggregate_data(df, group_by_cols, output_file)